# Retail Gasoline Price Forecasting
### U.S. Weekly Retail Gas Prices — Trend, Seasonality, and Structural Breaks

**Project 22 of 50 - Time Series Forecasting Portfolio**

## Project Overview

U.S. weekly retail gasoline prices capture 27 years of energy market history: the low-price 1990s, the 2008 oil shock to $4/gallon, the COVID-19 crash, and the 2021 recovery. This notebook forecasts **national average regular conventional gasoline retail prices** using StatsForecast.

Gasoline prices are **not white noise** - they exhibit:
- Slow mean reversion after shocks
- Weak annual seasonality (summer driving vs. winter)
- Structural breaks from OPEC decisions, recessions, and hurricanes

| Attribute | Value |
|---|---|
| **Dataset** | `mruanova/us-gasoline-and-diesel-retail-prices-19952021` |
| **Target** | U.S. national average retail price ($/gallon) |
| **Date column** | Weekly date |
| **Frequency** | Weekly |
| **TS Library** | StatsForecast |
| **Period** | 1995 - 2021 |

## Learning Objectives

1. Handle a **non-stationary, trending price series** with structural breaks
2. Detect **OPEC/macroeconomic shock events** in the time series
3. Apply StatsForecast AutoARIMA, AutoETS, and Theta to weekly price data
4. Use the **Crude oil price spread** as an informal reference against retail prices
5. Understand why forecasting price *levels* vs. price *changes* are very different tasks
6. Evaluate with MAPE and RMSE on a volatile price series

## Problem Statement

Given 25 years of weekly U.S. retail gasoline prices, forecast the next **8 weeks** (2 months) of national average prices.

This is the horizon used by fuel retailers for hedging decisions and by fleet operators for fuel cost budgeting.

## Why This Matters

- U.S. consumers spend ~$300 billion/year on retail gasoline
- Fleet operators (UPS, Amazon, logistics) pay fuel surcharges tied to weekly EIA price benchmarks
- Supply chain management: retailers hedge forward contracts based on price forecasts
- Government policy: gasoline price forecasts feed into consumer price index (CPI) projections
- Utilities and municipalities use fuel price forecasts for fleet cost planning

## Dataset Overview

**Source:** Kaggle - `mruanova/us-gasoline-and-diesel-retail-prices-19952021`

**Columns include:**
| Column | Description |
|---|---|
| Weekly date column | Timestamp (weekly, typically Mondays) |
| `U.S.` | **TARGET** - National avg regular conventional gasoline ($/gal) |
| Regional prices | East Coast, New England, Central Atlantic, Lower Atlantic, Midwest, Gulf Coast, Rocky Mountain, West Coast |
| `U.S. #2 Diesel Retail Prices` | Diesel national average |

**Period:** ~1995 to 2021 | **Frequency:** Weekly | **Observations:** ~1,350 rows

## Dataset Source & License

- **Kaggle slug:** `mruanova/us-gasoline-and-diesel-retail-prices-19952021`
- **License:** Public domain (source: U.S. Energy Information Administration - EIA)
- **EIA Primary source:** https://www.eia.gov/petroleum/gasprices/
- Auto-detects column names at runtime for robustness

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
            "scikit-learn","lazypredict","flaml","statsforecast"]:
    try: __import__(pkg.split("[")[0].replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from statsforecast import StatsForecast
from statsforecast.models import (AutoARIMA, AutoETS, SeasonalNaive, Theta,
                                   NaiveVariance, RandomWalkWithDrift)

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK")

## Configuration & Constants

In [ ]:
PROJECT      = "Retail Gasoline Price Forecasting"
KAGGLE_SLUG  = "mruanova/us-gasoline-and-diesel-retail-prices-19952021"
TARGET       = "U.S."    # will be auto-detected if not found
DATE_COL     = "Date"    # auto-detected
SEASON       = 52        # weekly: ~52 weeks per year
HORIZON      = 8         # 8-week forecast (2 months)
TEST_SIZE    = HORIZON * 4   # 32 weeks
VAL_SIZE     = HORIZON * 4
RANDOM_STATE = 42
FLAML_BUDGET = 120

DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)
print(f"Horizon: {HORIZON} weeks  |  Season: {SEASON} weeks/year  |  Test: {TEST_SIZE} weeks")

## Kaggle Credential Check

In [ ]:
cred = (os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or
        os.environ.get("KAGGLE_API_TOKEN") or
        (pathlib.Path.home()/".kaggle"/"kaggle.json").exists())
if cred: print("Kaggle credentials found.")
else:
    print("WARNING: No credentials. Set KAGGLE_USERNAME + KAGGLE_KEY env vars.")

## Dataset Download & Loading

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Data at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

csv_files = list(data_path.rglob("*.csv"))
print("Files:"); [print(f"  {f.name}  {f.stat().st_size/1e3:.0f}KB") for f in csv_files]

In [ ]:
if not csv_files:
    raise FileNotFoundError("No CSV files found. Check Kaggle credentials and slug.")

# Try to find the main gasoline prices file
main_file = next((f for f in csv_files if "gasoline" in f.name.lower() or "retail" in f.name.lower()),
                  csv_files[0])
print(f"Loading: {main_file.name}")
df_raw = pd.read_csv(main_file)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")

# Auto-detect date and target
date_cands   = [c for c in df_raw.columns if "date" in c.lower() or "week" in c.lower()]
target_cands = [c for c in df_raw.columns if c.strip() == "U.S." or "u.s." in c.lower()]
if not target_cands:
    numeric_cols = df_raw.select_dtypes(include="number").columns.tolist()
    target_cands = numeric_cols[:1]

DATE_COL = date_cands[0]   if date_cands   else df_raw.columns[0]
TARGET   = target_cands[0] if target_cands else df_raw.columns[1]
print(f"Date column   : '{DATE_COL}'")
print(f"Target column : '{TARGET}'")
df_raw.head(3)

## Data Validation Checks

In [ ]:
print("DATA VALIDATION REPORT")
print("="*55)
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL], errors="coerce")
print(f"Date range : {df_raw[DATE_COL].min().date()} -> {df_raw[DATE_COL].max().date()}")
print(f"Rows       : {len(df_raw):,}  (~{len(df_raw)/52:.1f} years of weekly data)")

miss = df_raw[TARGET].isna().sum()
print(f"Target NaN : {miss}")
print(f"Price range: ${df_raw[TARGET].min():.3f} - ${df_raw[TARGET].max():.3f} per gallon")

# Check if price looks like USD per gallon (should be 0.8 - 6.0 range roughly)
median_price = df_raw[TARGET].median()
if 0.8 < median_price < 7.0:
    print(f"Median price: ${median_price:.3f}/gal - looks like USD/gallon")
else:
    print(f"Median: {median_price:.4f} - check units!")
print(f"Duplicates : {df_raw.duplicated().sum()}")

## Exploratory Data Analysis

In [ ]:
ts_df = (df_raw[[DATE_COL, TARGET]].dropna()
               .sort_values(DATE_COL)
               .drop_duplicates(DATE_COL)
               .rename(columns={DATE_COL:"ds", TARGET:"y"})
               .reset_index(drop=True))

print(f"Series: {len(ts_df)} weekly observations")

# Annotate major events
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_df["ds"], y=ts_df["y"], name="US Regular Gasoline",
                          line=dict(color="steelblue", width=1.5)))

events = {
    "2008 Oil Shock": ("2008-07-07", 4.10),
    "2008 Crash":     ("2009-01-05", 1.60),
    "2011 Recovery":  ("2011-05-02", 4.02),
    "COVID-19 Crash": ("2020-04-27", 1.65),
}
for label, (date, y_pos) in events.items():
    try:
        fig.add_vline(x=date, line_dash="dot", line_color="red", opacity=0.5)
        fig.add_annotation(x=date, y=y_pos, text=label, showarrow=False,
                           textangle=-90, font=dict(size=9, color="red"))
    except: pass

fig.update_layout(title="U.S. Weekly Retail Gasoline Price 1995-2021 ($/gallon)",
                  xaxis_title="Date", yaxis_title="Price (USD/gallon)",
                  template="plotly_white")
fig.show()

In [ ]:
# Seasonal analysis
ts_df["year"]  = ts_df["ds"].dt.year
ts_df["week"]  = ts_df["ds"].dt.isocalendar().week.astype(int)
ts_df["month"] = ts_df["ds"].dt.month

# Monthly average across all years
monthly = ts_df.groupby("month")["y"].mean()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
months_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
axes[0].bar(months_names[:len(monthly)], monthly.values, color="steelblue", alpha=0.8)
axes[0].set_title("Average Price by Month (all years)")
axes[0].set_ylabel("Mean Price ($/gal)")

# Year-over-year lines (last 10 years)
recent_years = ts_df[ts_df["year"] >= ts_df["year"].max()-10]
for yr, grp in recent_years.groupby("year"):
    axes[1].plot(grp["week"], grp["y"], alpha=0.5, label=str(yr), linewidth=1)
axes[1].set_title("Year-by-Year Weekly Price (last 10 years)")
axes[1].set_xlabel("Week of Year"); axes[1].set_ylabel("Price ($/gal)")
axes[1].legend(fontsize=7, ncol=3)
plt.tight_layout(); plt.show()

In [ ]:
# First difference - stationarity transformation
ts_df["ydiff"] = ts_df["y"].diff()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ts_df["ds"].iloc[1:], ts_df["ydiff"].iloc[1:], lw=0.8, color="coral")
axes[0].axhline(0, color="black", lw=0.5, ls="--")
axes[0].set_title("First Difference of Gasoline Price")
axes[0].set_ylabel("Weekly Price Change ($/gal)")

plot_acf(ts_df["ydiff"].dropna(), lags=52, ax=axes[1], title="ACF of First Differences (52 weeks)")
plt.tight_layout(); plt.show()
print(f"Max single-week spike: ${ts_df['ydiff'].abs().max():.4f}")

In [ ]:
# ADF test on levels and first differences
adf_level = adfuller(ts_df["y"].dropna(), maxlag=10, autolag="AIC")
adf_diff  = adfuller(ts_df["ydiff"].dropna(), maxlag=10, autolag="AIC")
print(f"ADF on levels      : stat={adf_level[0]:.4f}  p={adf_level[1]:.4f}  -> {'Stationary' if adf_level[1]<0.05 else 'Non-stationary'}")
print(f"ADF on first diff  : stat={adf_diff[0]:.4f}  p={adf_diff[1]:.4f}  -> {'Stationary' if adf_diff[1]<0.05 else 'Non-stationary'}")
print("Conclusion: AutoARIMA will handle differencing automatically (d=1 expected)")

## Train / Validation / Test Split

In [ ]:
n = len(ts_df)
ts_test     = ts_df.iloc[n-TEST_SIZE:].copy()
ts_val      = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_train    = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_trainval = ts_df.iloc[:n-TEST_SIZE].copy()

print(f"Train:  {len(ts_train):,} weeks ({ts_train['ds'].min().date()} -> {ts_train['ds'].max().date()})")
print(f"Val:    {len(ts_val):,} weeks  ({ts_val['ds'].min().date()} -> {ts_val['ds'].max().date()})")
print(f"Test:   {len(ts_test):,} weeks  ({ts_test['ds'].min().date()} -> {ts_test['ds'].max().date()})")
assert ts_train["ds"].max() < ts_val["ds"].min() < ts_test["ds"].min()
print("Clean temporal split.")

## Preprocessing

In [ ]:
def preprocess(df_in):
    df = df_in.copy().sort_values("ds").drop_duplicates("ds").reset_index(drop=True)
    full_range = pd.date_range(df["ds"].min(), df["ds"].max(), freq="W-MON")
    df = df.set_index("ds").reindex(full_range).reset_index().rename(columns={"index":"ds"})
    miss = df["y"].isna().sum()
    if miss: df["y"] = df["y"].interpolate("linear"); print(f"  Interpolated {miss} missing weeks")
    return df

ts_train    = preprocess(ts_train)
ts_val      = preprocess(ts_val)
ts_test     = preprocess(ts_test)
ts_trainval = preprocess(ts_trainval)
print("Preprocessing done.")

## Feature Engineering

Gasoline prices exhibit **strong momentum** (price changes are slow-moving) and **reversal after spikes**. Key features:
- Lag 1-4 weeks: immediate price momentum
- Lag 52 weeks: same week last year (annual seasonality)
- Rolling mean / std: detect trend changes
- First difference: stationary transformation
- Log ratio: proportional change (returns perspective)

In [ ]:
def make_gas_features(df_in):
    df = df_in.copy()
    for lag in [1, 2, 3, 4, 8, 12, 26, 52]:
        if lag < len(df): df[f"lag_{lag}"] = df["y"].shift(lag)
    df["ydiff_1"]      = df["y"].diff(1)
    df["ydiff_4"]      = df["y"].diff(4)
    df["log_ret_1"]    = np.log(df["y"]).diff(1)
    df["roll_mean_4"]  = df["y"].shift(1).rolling(4).mean()
    df["roll_mean_12"] = df["y"].shift(1).rolling(12).mean()
    df["roll_std_4"]   = df["y"].shift(1).rolling(4).std()
    df["week"]         = df["ds"].dt.isocalendar().week.astype(int)
    df["month_sin"]    = np.sin(2*np.pi*df["ds"].dt.month/12)
    df["month_cos"]    = np.cos(2*np.pi*df["ds"].dt.month/12)
    return df

ts_all  = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_feat = make_gas_features(ts_all)
feat_cols = [c for c in ts_feat.columns if c not in ["ds","y","year","week_iso","month","ydiff"]]
split1 = len(ts_train); split2 = split1 + len(ts_val)
train_f = ts_feat.iloc[:split1].dropna().copy()
val_f   = ts_feat.iloc[split1:split2].dropna().copy()
test_f  = ts_feat.iloc[split2:].dropna().copy()
# final feat_cols from actual columns
feat_cols = [c for c in train_f.columns if c not in ["ds","y"]]
print(f"Features ({len(feat_cols)}): {feat_cols}")

## Baseline Approaches

In [ ]:
results = []
def evaluate(actual, pred, name):
    a,p = np.array(actual).flatten(), np.array(pred).flatten()
    n = min(len(a),len(p)); a,p = a[:n],p[:n]
    mae  = mean_absolute_error(a,p)
    rmse = np.sqrt(mean_squared_error(a,p))
    mape = np.mean(np.abs((a-p)/np.where(np.abs(a)>0.01, a, np.nan)))*100
    print(f"{name:<44s} MAE=${mae:.4f}  RMSE=${rmse:.4f}  MAPE={mape:.2f}%")
    results.append({"model":name,"MAE":mae,"RMSE":rmse,"MAPE":mape})

last_val = ts_trainval["y"].iloc[-1]
evaluate(ts_test["y"], np.full(len(ts_test), last_val), "Naive (last price)")
sn = np.tile(ts_trainval["y"].iloc[-SEASON:].values, len(ts_test)//SEASON+1)[:len(ts_test)]
evaluate(ts_test["y"], sn, "Seasonal Naive (same week last year)")

# Random Walk with Drift (very common for prices)
mean_diff = ts_trainval["y"].diff().mean()
rwd = last_val + np.cumsum(np.full(len(ts_test), mean_diff))
evaluate(ts_test["y"], rwd, "Random Walk with Drift")

## LazyPredict Benchmark

In [ ]:
X_tr = train_f[feat_cols]; y_tr = train_f["y"]
X_va = val_f[feat_cols];   y_va = val_f["y"]
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lm, _ = lazy.fit(X_tr, X_va, y_tr, y_va)
    print(lm.head(12).to_string())
except Exception as e: print(f"LazyPredict: {e}")

## FLAML AutoML

In [ ]:
X_all = pd.concat([train_f, val_f])[feat_cols]
y_all = pd.concat([train_f, val_f])["y"]
X_te  = test_f[feat_cols]
flaml = AutoML()
flaml.fit(X_all, y_all, task="regression", metric="rmse",
          time_budget=FLAML_BUDGET, verbose=0, seed=RANDOM_STATE)
flaml_pred = flaml.predict(X_te)
print(f"Best: {flaml.best_estimator}")
evaluate(test_f["y"], flaml_pred, f"FLAML ({flaml.best_estimator})")

## StatsForecast - Time Series Library

**Why StatsForecast for price forecasting?**

Gasoline prices are non-stationary (d=1) with weak seasonal structure. StatsForecast models:
1. **AutoARIMA(m=52)**: seasonal ARIMA, will likely select d=1, D=0 (prices are I(1) not seasonally integrated)
2. **AutoETS(m=52)**: error/trend/season - likely selects ETS(M,A,N) or ETS(A,A,N)
3. **Theta**: robust to structural breaks; decomposes series into trend + cyclical component
4. **RandomWalkWithDrift**: the classic price model (Efficient Market Hypothesis baseline)

In [ ]:
sf_train = ts_trainval[["ds","y"]].copy()
sf_train.insert(0, "unique_id", "US_gasoline")

models = [
    AutoARIMA(m=52, max_p=3, max_d=2, max_q=3, max_P=1, max_D=1, max_Q=1),
    AutoETS(season_length=52, model="ZZZ"),
    Theta(season_length=52),
    SeasonalNaive(season_length=52),
]

sf = StatsForecast(models=models, freq="W-MON", n_jobs=-1)
try:
    sf_pred = sf.forecast(df=sf_train, h=HORIZON * 4)  # forecast full test period
    print("StatsForecast forecast generated:")
    print(sf_pred.head(5))
except Exception as e:
    print(f"StatsForecast error: {e}")
    sf_pred = None

In [ ]:
if sf_pred is not None:
    pred_df = sf_pred.reset_index(drop=True)
    for col in [c for c in pred_df.columns if c not in ["unique_id","ds"]]:
        try:
            n = min(len(ts_test), len(pred_df))
            evaluate(ts_test["y"].values[:n], pred_df[col].values[:n], f"SF-{col}")
        except Exception as e: print(f"  {col}: {e}")

In [ ]:
# Forecast plot
fig = go.Figure()
# Show last 52 weeks of training + test
context = ts_trainval.iloc[-52:]
fig.add_trace(go.Scatter(x=context["ds"], y=context["y"],
                          name="Historical (last year)", line=dict(color="grey", width=1.5, dash="dot")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
                          name="Actual Test", line=dict(color="black", width=2)))
colors = ["steelblue","darkorange","green","purple"]
if sf_pred is not None:
    pred_df = sf_pred.reset_index(drop=True)
    for col, clr in zip([c for c in pred_df.columns if c not in ["unique_id","ds"]], colors):
        fig.add_trace(go.Scatter(x=ts_test["ds"].iloc[:len(pred_df)], y=pred_df[col].values,
                                  name=col, line=dict(color=clr, dash="dash")))
fig.update_layout(title="U.S. Retail Gas Price - StatsForecast Models",
                  xaxis_title="Date", yaxis_title="Price ($/gallon)",
                  template="plotly_white"); fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("ALL MODELS (ranked by RMSE)"); print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("
TOP 3:"); print(top3.to_string(index=False))

fig = px.bar(results_df, x="model", y="RMSE",
             title="Gas Price Forecast Model Comparison",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-40, template="plotly_white"); fig.show()

## Error Analysis

In [ ]:
if len(test_f) > 0:
    actual = test_f["y"].values
    pred   = flaml.predict(test_f[feat_cols])
    errors = actual - pred

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].hist(errors, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
    axes[0].axvline(0, color="red", ls="--"); axes[0].set_title("Residual Distribution ($/gal)")

    axes[1].plot(range(len(errors)), errors, color="coral", alpha=0.7)
    axes[1].axhline(0, color="red", ls="--")
    axes[1].set_title("Residuals Over Forecast Horizon")
    axes[1].set_xlabel("Week ahead"); axes[1].set_ylabel("Error ($/gal)")

    axes[2].scatter(actual, pred, s=30, alpha=0.6, color="steelblue")
    lo,hi = min(actual.min(),pred.min()), max(actual.max(),pred.max())
    axes[2].plot([lo,hi],[lo,hi],"r--")
    axes[2].set_title("Actual vs Predicted")
    axes[2].set_xlabel("Actual ($/gal)"); axes[2].set_ylabel("Predicted ($/gal)")
    plt.tight_layout(); plt.show()

    bias = errors.mean()
    print(f"Mean Bias: ${bias:+.4f}/gal  ({'over' if bias < 0 else 'under'}-forecast)")

## Interpretation & Insights

1. **The Naive (last price) model is surprisingly competitive** - gasoline prices are highly autocorrelated (I(1)), so last week's price is a strong predictor of this week's price
2. **FLAML with lag_1 and lag_4 typically wins** on the training period, but the real test is structural breaks (recessions, pandemics) that the model was never trained on
3. **Theta handles structural breaks better** than ARIMA because it decomposes the series into a long-run trend and a mean-reverting cycle
4. **Seasonal component is weak for gasoline** vs. heating oil - summer driving season adds ~$0.10-0.20/gallon; most models identify this but it is a small signal
5. **MAPE is reliable here** unlike quantity-based series - prices are always strictly positive

## Limitations

1. **No crude oil input**: WTI crude oil is the primary cost driver; without it, models are forecasting the effect without the cause
2. **Structural breaks uncaptured**: the 2008 and 2020 crashes are impossible to predict from history alone; rolling-window models will perform poorly through these events
3. **National average masks regional variation**: West Coast prices ($1-2/gal premium) behave differently from Gulf Coast prices
4. **Refinery capacity events**: hurricanes, plant outages, and OPEC output cuts are exogenous shocks that dominate short-term price movements
5. **Weekly frequency is too coarse for trading**: day-traders use daily or intraday data; weekly is relevant for fleet budgeting and derivatives hedging

## How to Improve This Project

1. **Add WTI crude oil price**: join the EIA weekly crude oil price; `crack spread = gas_price - crude_price/42` is a stable indicator of refinery margins
2. **GARCH model for volatility**: gasoline prices exhibit volatility clustering - ARCH/GARCH models capture this when accuracy of variance matters as much as point forecasts
3. **Event dummy variables**: add OPEC meeting dates, US hurricane season indicator (Sept-Oct), and recession indicators (NBER dates)
4. **Vector Autoregression (VAR)**: model gasoline + diesel + crude jointly to capture cross-commodity spillovers
5. **Forecast evaluation by regime**: separately report accuracy during stable periods vs. shock periods (variance > 1.5 sigma)

## Production Considerations

1. **Weekly pipeline**: pull EIA data every Monday at noon (EIA publishes Monday afternoons); generate 8-week ahead forecast
2. **Fuel cost budgeting API**: expose price forecasts to fleet management system for weekly fuel surcharge calculations
3. **Scenario generation**: produce three scenarios (high/base/low) using historical percentile shocks applied to the base forecast
4. **Model drift detection**: track 4-week rolling MAPE; if MAPE > 5%, trigger manual review and model refresh
5. **Audit trail**: log every forecast with the week it was made, model version, and eventual actuals for regulatory price review purposes

## Common Mistakes to Avoid

1. **Seasonal differencing at m=52 for gas prices**: gas prices are NOT seasonally integrated (D=0 is correct); seasonal differencing destroys the signal
2. **Confusing price levels with quantity**: gasoline price series is I(1) and needs d=1 differencing; treating it as stationary produces autocorrelated residuals
3. **Using RMSE without context**: $0.10/gallon RMSE is excellent in a $2/gal environment but terrible in a $4/gal environment; always also report MAPE
4. **Training on the full 27-year series without acknowledging regime changes**: pre-2005 low-price regime differs structurally from post-2005; consider using only post-2010 data for current regime forecasting
5. **Ignoring fat tails**: gas price changes have heavier tails than normal - use Huber loss or quantile regression if outlier robustness matters

## Mini Challenge / Exercises

1. **Add crude oil**: download weekly WTI crude oil prices from quandl or FRED (free API), compute the crack spread, add it as a feature. Report RMSE delta.
2. **Regime detection**: use a rolling 52-week standard deviation to identify high-volatility vs low-volatility regimes. Train separate models per regime.
3. **Regional comparison**: load all regional price columns (East Coast, Gulf Coast, West Coast). Which region is hardest to forecast?
4. **GARCH volatility**: fit an ARCH(1) model to the first differences. Plot the time-varying conditional variance - when did it peak?
5. **Scenario forecast**: generate a +20% crude oil shock scenario. How does it propagate through the gas price forecast?

## Final Summary & Key Takeaways

### What We Did
- Downloaded 27 years of weekly U.S. retail gasoline prices from the EIA via Kaggle
- Characterised the I(1) non-stationary nature (requires first differencing for stationarity)
- Annotated key structural breaks: 2008 oil shock, COVID-19 crash
- Built momentum and seasonal lag features + rolling statistics
- Benchmarked with LazyPredict (includes Random Forest, Gradient Boosting, Ridge, etc.)
- Ran FLAML AutoML (120s) for optimal ML baseline
- Fitted StatsForecast: AutoARIMA(m=52), AutoETS(m=52), Theta, SeasonalNaive(52)
- Selected Top 3 by RMSE; analysed residuals and bias

### Key Takeaways
1. **Naive (last price) is the hardest baseline to beat** for commodity prices - the process is close to a random walk
2. **AutoARIMA(d=1, D=0)** is the correct choice - gasoline prices are I(1), not seasonally integrated
3. **Theta handles regime changes better** than model-fit-based approaches
4. **Crude oil price is the missing causal feature** - without it, all models are forecasting symptoms, not causes
5. **Short horizons (1-4 weeks) are more accurate** than long horizons (8-12 weeks) for price series with structural breaks

---
*Educational notebook - part of the 50-project Time Series Forecasting portfolio.*